#1. Apache Spark Architecture

Apache Spark follows a master-worker architecture that consists of three main components:

###Driver

The Driver is the main program that controls the Spark application. It creates the SparkSession, divides the application into tasks, schedules those tasks, and collects the final results from the executors.

###Cluster Manager

The Cluster Manager is responsible for managing the resources of the cluster. It allocates CPU and memory to different Spark applications and launches executors on worker nodes. Spark supports different cluster managers such as Standalone, Hadoop YARN, Kubernetes, and Apache Mesos.

###Executors

Executors are worker processes that execute the tasks assigned by the Driver. They perform computations, store intermediate data in memory or disk, and send the results back to the Driver.

#2. Execution Modes

Spark applications can run in two execution modes:

###Client Mode

In Client Mode, the Driver runs on the local machine from which the application is submitted. This mode is commonly used for development and testing.

###Cluster Mode

In Cluster Mode, the Driver runs inside the cluster. This mode is preferred for production because the application continues to run even if the client machine disconnects.

#3. Lazy Evaluation
Spark uses Lazy Evaluation, which means that transformations such as filter(), select(), and withColumn() are not executed immediately. Instead, Spark records these operations and waits until an action like show(), count(), or collect() is called. This allows Spark to optimize the execution plan, reduce unnecessary computations, and improve overall performance.

#4. Directed Acyclic Graph (DAG) / Lineage Graph

Spark builds a Directed Acyclic Graph (DAG) of all transformations applied to the data. The DAG represents the sequence of operations required to produce the final result. It enables Spark to optimize execution and provides fault tolerance by recomputing only the lost partitions if a worker node fails, instead of rerunning the entire application.

#5. Transformations and Actions

Spark operations are divided into two categories:

###Transformations

Transformations create a new DataFrame or RDD without immediately executing the computation. They are evaluated lazily.

####Examples:

select()
filter()
withColumn()
withColumnRenamed()
drop()

###Actions

Actions trigger the execution of all pending transformations and return the result or write it to storage.

####Examples:

show()
count()
collect()
first()
write()

#6. Shuffle

Shuffle is the process of redistributing data across executors during wide transformations such as groupBy(), join(), distinct(), and orderBy(). Since data is transferred over the network and may be written to disk, shuffle is one of the most expensive operations in Spark. Minimizing unnecessary shuffle operations helps improve application performance.

#7. Predicate Pushdown

Predicate Pushdown is an optimization technique used mainly with Parquet files. Spark pushes filter conditions down to the storage layer, allowing only the required rows and columns to be read into memory. This reduces disk I/O, memory usage, and query execution time, making data processing more efficient.

#7. Predicate Pushdown

Predicate Pushdown is an optimization technique used mainly with Parquet files. Spark pushes filter conditions down to the storage layer, allowing only the required rows and columns to be read into memory. This reduces disk I/O, memory usage, and query execution time, making data processing more efficient.

#8. CSV
1. Row-based file format.
2. Human-readable and easy to edit.
3. Does not store schema information.
4. Larger file size.
5. Slower for analytical processing.

# Parquet
1. Columnar file format.
2. Stores schema information.
3. Compressed and occupies less storage.
4. Reads only required columns.
5. Faster for analytical workloads.

Parquet is generally preferred for big data analytics because it provides better compression, faster query execution, and supports advanced optimizations like Predicate Pushdown.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6Assignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [3]:
from google.colab import files

uploaded = files.upload()

Saving dataset.csv to dataset.csv


In [4]:
import os

print(os.listdir())

['.config', 'dataset.csv', 'sample_data']


In [5]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("dataset.csv")

In [6]:
df.show(5)

+-------+----------+-----------+-----+---------+------+------+--------+
|user_id|product_id|   category|price|   status|amount|region|priority|
+-------+----------+-----------+-----+---------+------+------+--------+
|    1.0|       101|Electronics|  450|  Pending|   450| South|    High|
|    2.0|       102|      Books|  650|Completed|   650|  West|    High|
|    3.0|       103|Electronics|  650|Completed|   650| South|     Low|
|    4.0|       104|       Toys|  450|Completed|   450|  West|    High|
|    5.0|       105|  Furniture| 3000|  Pending|  3000| North|    High|
+-------+----------+-----------+-----+---------+------+------+--------+
only showing top 5 rows


In [7]:
df.printSchema()

root
 |-- user_id: double (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



In [8]:
df.count()

50

In [9]:
df.select("product_id","price").show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101|  450|
|       102|  650|
|       103|  650|
|       104|  450|
|       105| 3000|
|       106| 1500|
|       107|  650|
|       108| 1500|
|       109| 2500|
|       110| 1200|
|       111|  450|
|       112|  650|
|       113|  800|
|       114| 4500|
|       115|  800|
|       116|  999|
|       117| 1500|
|       118| 3000|
|       119| 1800|
|       120|  999|
+----------+-----+
only showing top 20 rows


In [10]:
electronics = df.filter(df.category=="Electronics")

electronics.show()

+-------+----------+-----------+-----+---------+------+------+--------+
|user_id|product_id|   category|price|   status|amount|region|priority|
+-------+----------+-----------+-----+---------+------+------+--------+
|    1.0|       101|Electronics|  450|  Pending|   450| South|    High|
|    3.0|       103|Electronics|  650|Completed|   650| South|     Low|
|   11.0|       111|Electronics|  450|Completed|   450|  East|    High|
|   17.0|       117|Electronics| 1500|  Pending|  1500|  East|    High|
|   25.0|       125|Electronics| 4500|Completed|  4500|  East|     Low|
|   28.0|       128|Electronics| 4500|  Pending|  4500| South|    High|
|   30.0|       130|Electronics|  650|  Pending|   650|  East|    High|
|   31.0|       131|Electronics|  999|Completed|   999| North|     Low|
|   38.0|       138|Electronics| 4500|Completed|  4500| South|    High|
|   39.0|       139|Electronics| 1500|Completed|  1500| South|  Medium|
|   43.0|       143|Electronics| 4500|Completed|  4500| North|  

In [11]:
completed = df.filter(
    (df.status=="Completed") &
    (df.amount>1000)
)

completed.show()

+-------+----------+-----------+-----+---------+------+------+--------+
|user_id|product_id|   category|price|   status|amount|region|priority|
+-------+----------+-----------+-----+---------+------+------+--------+
|    9.0|       109|  Furniture| 2500|Completed|  2500|  West|    High|
|   14.0|       114|    Clothes| 4500|Completed|  4500| South|     Low|
|   21.0|       121|  Furniture| 1500|Completed|  1500| South|     Low|
|   25.0|       125|Electronics| 4500|Completed|  4500|  East|     Low|
|   33.0|       133|  Furniture| 2500|Completed|  2500|  East|     Low|
|   34.0|       134|       Toys| 1800|Completed|  1800| South|     Low|
|   38.0|       138|Electronics| 4500|Completed|  4500| South|    High|
|   39.0|       139|Electronics| 1500|Completed|  1500| South|  Medium|
|   NULL|       141|  Furniture| 1800|Completed|  1800| North|    High|
|   43.0|       143|Electronics| 4500|Completed|  4500| North|  Medium|
|   45.0|       145|       Toys| 2200|Completed|  2200|  West|  

In [12]:
north = df.filter(
    (df.region=="North") |
    (df.priority=="High")
)

north.show()

+-------+----------+-----------+-----+---------+------+------+--------+
|user_id|product_id|   category|price|   status|amount|region|priority|
+-------+----------+-----------+-----+---------+------+------+--------+
|    1.0|       101|Electronics|  450|  Pending|   450| South|    High|
|    2.0|       102|      Books|  650|Completed|   650|  West|    High|
|    4.0|       104|       Toys|  450|Completed|   450|  West|    High|
|    5.0|       105|  Furniture| 3000|  Pending|  3000| North|    High|
|   NULL|       106|  Furniture| 1500|  Pending|  1500| South|    High|
|    7.0|       107|    Clothes|  650|Completed|   650|  West|    High|
|    8.0|       108|    Clothes| 1500|  Pending|  1500| North|     Low|
|    9.0|       109|  Furniture| 2500|Completed|  2500|  West|    High|
|   11.0|       111|Electronics|  450|Completed|   450|  East|    High|
|   13.0|       113|    Clothes|  800|  Pending|   800|  East|    High|
|   16.0|       116|       Toys|  999|  Pending|   999| North|  

In [13]:
df = df.withColumnRenamed(
    "price",
    "base_price"
)

df.show(5)

+-------+----------+-----------+----------+---------+------+------+--------+
|user_id|product_id|   category|base_price|   status|amount|region|priority|
+-------+----------+-----------+----------+---------+------+------+--------+
|    1.0|       101|Electronics|       450|  Pending|   450| South|    High|
|    2.0|       102|      Books|       650|Completed|   650|  West|    High|
|    3.0|       103|Electronics|       650|Completed|   650| South|     Low|
|    4.0|       104|       Toys|       450|Completed|   450|  West|    High|
|    5.0|       105|  Furniture|      3000|  Pending|  3000| North|    High|
+-------+----------+-----------+----------+---------+------+------+--------+
only showing top 5 rows


In [15]:
from pyspark.sql.functions import col
df = df.withColumn(
    "base_price",
    col("base_price").cast("double")
)

df.printSchema()

root
 |-- user_id: double (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



In [18]:
from pyspark.sql.functions import col

df = df.withColumn(
    "user_id",
    col("user_id").cast("int")
)

In [19]:
df = df.withColumn(
    "final_price",
    col("base_price")*1.18
)

df.show(10)

+-------+----------+-----------+----------+---------+------+------+--------+-----------+
|user_id|product_id|   category|base_price|   status|amount|region|priority|final_price|
+-------+----------+-----------+----------+---------+------+------+--------+-----------+
|      1|       101|Electronics|     450.0|  Pending|   450| South|    High|      531.0|
|      2|       102|      Books|     650.0|Completed|   650|  West|    High|      767.0|
|      3|       103|Electronics|     650.0|Completed|   650| South|     Low|      767.0|
|      4|       104|       Toys|     450.0|Completed|   450|  West|    High|      531.0|
|      5|       105|  Furniture|    3000.0|  Pending|  3000| North|    High|     3540.0|
|      7|       107|    Clothes|     650.0|Completed|   650|  West|    High|      767.0|
|      8|       108|    Clothes|    1500.0|  Pending|  1500| North|     Low|     1770.0|
|      9|       109|  Furniture|    2500.0|Completed|  2500|  West|    High|     2950.0|
|     10|       110| 

In [20]:
df = df.filter(
    df.user_id.isNotNull()
)

df.show()

+-------+----------+-----------+----------+---------+------+------+--------+-----------+
|user_id|product_id|   category|base_price|   status|amount|region|priority|final_price|
+-------+----------+-----------+----------+---------+------+------+--------+-----------+
|      1|       101|Electronics|     450.0|  Pending|   450| South|    High|      531.0|
|      2|       102|      Books|     650.0|Completed|   650|  West|    High|      767.0|
|      3|       103|Electronics|     650.0|Completed|   650| South|     Low|      767.0|
|      4|       104|       Toys|     450.0|Completed|   450|  West|    High|      531.0|
|      5|       105|  Furniture|    3000.0|  Pending|  3000| North|    High|     3540.0|
|      7|       107|    Clothes|     650.0|Completed|   650|  West|    High|      767.0|
|      8|       108|    Clothes|    1500.0|  Pending|  1500| North|     Low|     1770.0|
|      9|       109|  Furniture|    2500.0|Completed|  2500|  West|    High|     2950.0|
|     10|       110| 

In [21]:
result = df.select(
    "product_id",
    "base_price",
    "final_price"
)

result.show()

+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|       101|     450.0|      531.0|
|       102|     650.0|      767.0|
|       103|     650.0|      767.0|
|       104|     450.0|      531.0|
|       105|    3000.0|     3540.0|
|       107|     650.0|      767.0|
|       108|    1500.0|     1770.0|
|       109|    2500.0|     2950.0|
|       110|    1200.0|     1416.0|
|       111|     450.0|      531.0|
|       112|     650.0|      767.0|
|       113|     800.0|      944.0|
|       114|    4500.0|     5310.0|
|       115|     800.0|      944.0|
|       116|     999.0|    1178.82|
|       117|    1500.0|     1770.0|
|       118|    3000.0|     3540.0|
|       119|    1800.0|     2124.0|
|       120|     999.0|    1178.82|
|       121|    1500.0|     1770.0|
+----------+----------+-----------+
only showing top 20 rows


In [23]:
result.write \
.mode("overwrite") \
.option("header",True) \
.csv("output_csv")

In [24]:
result.write \
.mode("overwrite") \
.parquet("output_parquet")

In [25]:
parquet_df = spark.read.parquet(
    "output_parquet"
)

parquet_df.show()

+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|       101|     450.0|      531.0|
|       102|     650.0|      767.0|
|       103|     650.0|      767.0|
|       104|     450.0|      531.0|
|       105|    3000.0|     3540.0|
|       107|     650.0|      767.0|
|       108|    1500.0|     1770.0|
|       109|    2500.0|     2950.0|
|       110|    1200.0|     1416.0|
|       111|     450.0|      531.0|
|       112|     650.0|      767.0|
|       113|     800.0|      944.0|
|       114|    4500.0|     5310.0|
|       115|     800.0|      944.0|
|       116|     999.0|    1178.82|
|       117|    1500.0|     1770.0|
|       118|    3000.0|     3540.0|
|       119|    1800.0|     2124.0|
|       120|     999.0|    1178.82|
|       121|    1500.0|     1770.0|
+----------+----------+-----------+
only showing top 20 rows


In [26]:
!zip -r output_csv.zip output_csv
!zip -r output_parquet.zip output_parquet

  adding: output_csv/ (stored 0%)
  adding: output_csv/part-00000-8f3b9066-8401-4c87-ab1e-8c2decbd6f6c-c000.csv (deflated 73%)
  adding: output_csv/.part-00000-8f3b9066-8401-4c87-ab1e-8c2decbd6f6c-c000.csv.crc (stored 0%)
  adding: output_csv/_SUCCESS (stored 0%)
  adding: output_csv/._SUCCESS.crc (stored 0%)
  adding: output_parquet/ (stored 0%)
  adding: output_parquet/.part-00000-1b8fd848-8f30-4432-9653-4088eb401f29-c000.snappy.parquet.crc (stored 0%)
  adding: output_parquet/_SUCCESS (stored 0%)
  adding: output_parquet/part-00000-1b8fd848-8f30-4432-9653-4088eb401f29-c000.snappy.parquet (deflated 43%)
  adding: output_parquet/._SUCCESS.crc (stored 0%)


In [27]:
from google.colab import files

files.download("output_csv.zip")
files.download("output_parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>